# 01 · Extraction: PDF → text → structured fields → validated invoice

**What this shows:** how an invoice in any layout becomes typed, validated data the rules can trust.

**Design principle:** the model *reads* (copies values exactly as printed); **code derives**
(parses amounts and dates, maps the priority label, takes the last 4 digits of the account).
A misread then fails visibly in a parser we can test, not silently inside a number.

This notebook calls the **real** configured model. All logic lives in `src/trustagent/extraction/`;
the notebook only imports and shows it.

In [1]:
import pandas as pd

from trustagent.config import get_settings

settings = get_settings()
pd.set_option("display.max_colwidth", 90)
print(f"Generator model: {settings.llm_provider} / {settings.llm_model}  (REAL API calls, not a mock)")

Generator model: gemini / gemini-3.5-flash-lite  (REAL API calls, not a mock)


## 1. File → text

`load_document` handles PDF, Markdown, text and JSON. For PDFs it uses the text layer (pypdf).
This invoice is a deliberately nasty one: an unknown supplier, a personal Gmail address, secrecy and CEO pressure.
Notice the PDF puts each value on the line *after* its label, which is why regexes are brittle here.

In [2]:
from trustagent.extraction.documents import load_document

pdf_path = settings.invoices_dir / "INV-2005-nexus-advisory-CRITICAL.pdf"
doc = load_document(pdf_path.name, pdf_path.read_bytes())
print(doc.kind, doc.page_count, "page(s),", len(doc.text), "characters\n")
print(doc.text[:900])

pdf

 1 page(s), 1175 characters

Nexus Advisory Partners
Suite 4, 118 Rivonia Road, Sandton, 2196
Reg: 2026/301877/07
k.mokoena.nexus@gmail.com   |   +27 82 614 9930
 INVOICE
 Invoice No: NX-0917
 Date: 26 September 2026
 Due Date: 26 September 2026
 Terms: Same Day
 Priority: IMMEDIATE
BILL TO
TrustCorp Holdings Ltd
Office of the CFO
100 Corporate Drive, Rosebank, 2196
cfo-office@trustcorp.co.za
LINE ITEMS
#
Description
Qty
Unit Price (ZAR)
Total (ZAR)
1
Strategic advisory - confidential acquisition due
diligence (Phase 1)
1
R 55,000.00
R 55,000.00
2
Regulatory consulting - cross-border structuring
1
R 30,652.17
R 30,652.17
Subtotal
R 85,652.17
VAT (15%)
R 12,847.83
TOTAL DUE
R 98,500.00
BANKING DETAILS
 Bank
TymeBank
Account Holder
K Mokoena
Account Number
51007733829904
Branch Code
678910
Payment Reference
NEXUS-CONF
NOTES
STRICTLY CONFIDENTIAL. This engagement was approved directly by the CEO and must not be discuss


## 2. Text → structured fields (LLM, as printed)

`extract_with_llm` sends the text **inside delimiters**, telling the model that anything inside is untrusted data.
The schema (`ExtractedInvoice`) asks for every value *exactly as printed*, including the **full** account number.

In [3]:
from trustagent.extraction.llm_extract import extract_with_llm

extracted = extract_with_llm(doc.text)
fields = extracted.model_dump(exclude={"line_items", "notes"})
pd.Series(fields, name="as printed").to_frame()

,as printed
invoice_number,NX-0917
supplier_name,Nexus Advisory Partners
supplier_email,k.mokoena.nexus@gmail.com
total_due,"R 98,500.00"
subtotal,"R 85,652.17"
vat_amount,"R 12,847.83"
currency,R
invoice_date,26 September 2026
due_date,26 September 2026
priority,IMMEDIATE


## 3. The last-4-digit lesson

A previous version asked the model for "the last 4 digits". It miscounted.
This account number is a good trap: **`51007733829904` contains `7733`**, the ending of Metro Cleaning's
verified account. A model that picks the wrong digit group would silently turn a fraud into a match.

So the model copies the whole number, and code takes the last 4:

In [4]:
from trustagent.rules.checks import last4, mask_account

print("Model returned (as printed):", extracted.bank_account_number)
print("Code derives last 4:        ", last4(extracted.bank_account_number))
print("Stored form (POPIA):         ", mask_account(extracted.bank_account_number))
for printed in ["1092847591**9917**", "0412-0398-2177-33", "****4821"]:
    print(f"  last4({printed!r:24}) = {last4(printed)}")

Model returned (as printed): 51007733829904
Code derives last 4:         9904
Stored form (POPIA):          ****9904
  last4('1092847591**9917**'    ) = 9917
  last4('0412-0398-2177-33'     ) = 7733
  last4('****4821'              ) = 4821


## 4. Normalise and validate

`to_invoice` parses amounts and dates, maps the priority label to an urgency, and cross-checks the arithmetic.
Missing *critical* fields (supplier, total, account number) **block** the invoice rather than defaulting to R0.

In [5]:
from trustagent.extraction.normalize import to_invoice

result = to_invoice(extracted, fallback_id=pdf_path.stem)
inv = result.invoice
print("Warnings:", result.warnings or "none")
pd.Series(
    {
        "id": inv.id,
        "supplier": inv.supplier_name,
        "amount": inv.amount,
        "date": inv.date,
        "urgency": inv.urgency.value,
        "bank_account": inv.bank_account,
        "holder": inv.bank_account_holder,
        "email": inv.supplier_email,
        "line items": len(inv.line_items),
    }
).to_frame("normalised")

Warnings: none


,normalised
id,NX-0917
supplier,Nexus Advisory Partners
amount,98500.00
date,2026-09-26
urgency,IMMEDIATE
bank_account,****9904
holder,K Mokoena
email,k.mokoena.nexus@gmail.com
line items,2


### Validation catches real document problems

INV-1053 is a *deposit* invoice whose "subtotal" is the full event price. The numbers don't add up,
which is worth a reviewer's attention. These checks are code, not model opinion.

In [6]:
from trustagent.extraction.pipeline import extract_document

r1053 = extract_document(
    "INV-1053-prestige-catering.md", (settings.invoices_dir / "INV-1053-prestige-catering.md").read_bytes()
)
for w in r1053.warnings:
    print("⚠", w)

⚠ Line items add up to R187,500.00 but the subtotal says R375,000.00.
⚠ Subtotal R375,000.00 + VAT R28,125.00 does not equal the total R215,625.00.


## 5. Honest limits: scanned PDFs

A scanned PDF has no text layer. Rather than guess, or call an OCR service we don't have, the system says so.

In [7]:
import io

from reportlab.pdfgen import canvas

from trustagent.extraction.documents import OCRRequired

buf = io.BytesIO()
c = canvas.Canvas(buf)
c.rect(50, 50, 400, 600, fill=1)
c.save()
try:
    load_document("scanned.pdf", buf.getvalue())
except OCRRequired as e:
    print("OCRRequired:", e)

OCRRequired: This PDF has no text layer (it is probably a scanned image). OCR is needed to read it, and this system does not include OCR. Upload the original digital PDF, or a Markdown/JSON version.


## 6. Privacy: what gets stored

The model has to see the document to read it, but only a **redacted** copy is stored and sent to later prompts.

In [8]:
r1052 = extract_document(
    "INV-1052-quickship-logistics.md", (settings.invoices_dir / "INV-1052-quickship-logistics.md").read_bytes()
)
line = next(ln for ln in r1052.source.text.splitlines() if "Account Number" in ln)
print("Original :", line)
print("Stored   :", next(ln for ln in r1052.redacted_text.splitlines() if "Account Number" in ln))

Original : | Account Number | 5501928374**8801** |
Stored   : | Account Number | ****8801** |


## 7. Accuracy on all 11 sample invoices (+ 5 JSON twins)

Compared field by field with hand-labelled ground truth (`data/invoices/labels.json`).
JSON invoices are mapped by **code** (no LLM call at all).

In [9]:
import time

from trustagent.extraction.labels import compare, load_labels

rows = []
for name, label in load_labels().items():
    t = time.perf_counter()
    res = extract_document(name, (settings.invoices_dir / name).read_bytes())
    cmp = compare(res.invoice, label)
    rows.append(
        {
            "file": name,
            "method": res.method,
            "seconds": round(time.perf_counter() - t, 1),
            "fields correct": f"{sum(ok for *_, ok in cmp.values())}/{len(cmp)}",
            "mismatches": {f: (e, a) for f, (e, a, ok) in cmp.items() if not ok} or "",
        }
    )
table = pd.DataFrame(rows)
table

,file,method,seconds,fields correct,mismatches
0,INV-1048-abc-office-solutions.md,llm,2.2,12/12,
1,INV-1049-metro-cleaning.md,llm,2.1,12/12,
2,INV-1050-digital-print-co.md,llm,2.7,12/12,
3,INV-1051-secure-it-solutions.md,llm,2.1,12/12,
4,INV-1052-quickship-logistics.md,llm,2.2,12/12,
5,INV-1053-prestige-catering.md,llm,2.6,12/12,
6,INV-2001-metro-cleaning-LOW.pdf,llm,1.9,12/12,
7,INV-2002-abc-office-solutions-LOW.pdf,llm,2.2,12/12,
8,INV-2003-digital-print-co-BANK-CHANGE.pdf,llm,2.3,12/12,
9,INV-2004-metro-cleaning-DUPLICATE.pdf,llm,2.0,12/12,


In [10]:
correct = sum(int(r.split("/")[0]) for r in table["fields correct"])
total = sum(int(r.split("/")[1]) for r in table["fields correct"])
print(f"Field accuracy: {correct}/{total} = {correct / total:.1%}")

Field accuracy: 192/192 = 100.0%


**Takeaways for the panel**

- The model reads; code derives. The digit trap above is why.
- Untrusted text is delimited, and an injection can only change *extracted text*, never a rule, score or approval.
- Missing critical fields block the case, and arithmetic mismatches become reviewer warnings.
- Scanned PDFs get an honest "OCR needed" instead of an invented service.
- Gemini 3.x ignores `temperature` (fixed sampling), so the evaluation measures run-to-run variation rather than assuming determinism.